In [ ]:
import os
from pathlib import Path
# Ensure CWD is repo root so relative paths and `tools.*` imports resolve.
if Path.cwd().name == "notebooks":
    os.chdir("..")


# Annotation Error Detection on the decicontas.br Dataset

This notebook applies **Annotation Error Detection (AED)** techniques to the `decicontas.br` dataset, which contains decisions from the Rio Grande do Norte State Court of Accounts (TCE/RN), manually annotated via Label Studio with entities such as `MULTA`, `OBRIGACAO`, `RECOMENDACAO`, and `RESSARCIMENTO`.

The approach is inspired by the error-detection work on the LeNER-Br dataset, but with **significant improvements**:

1. **More robust model**: we use a Transformer fine-tuned for NER (token classification) instead of only an SGDClassifier over static embeddings.
2. **Model ensemble**: we combine predictions from a linear classifier (over BERT embeddings) and a fine-tuned Transformer for greater robustness.
3. **Format adaptation**: automatic conversion of Label Studio annotations (spans) to token-level BIO format.
4. **Contextual analysis**: rich display of the detected errors with expanded context.

The core technique is **Confident Learning**, implemented by the [Cleanlab](https://github.com/cleanlab/cleanlab) library, which identifies potentially mislabeled instances based on the probabilities predicted by a classifier.

# 1. Environment Setup

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import warnings
warnings.filterwarnings('ignore')

os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

from collections import defaultdict, Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report

from transformers import (
    AutoTokenizer, AutoModel, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
from datasets import Dataset, DatasetDict
from cleanlab.filter import find_label_issues

NUM_FOLDS_CV = 5
RANDOM_SEED = 43
EFFECTIVE_MAX_LENGTH = 512

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Usando device: {device}')

Usando device: mps


# 2. Dataset Loading and Conversion

The `decicontas.br` dataset uses Label Studio annotations (spans with `start`/`end`/`text`/`labels`). We need to convert them to token-level **BIO** (Beginning-Inside-Outside) format, which is the standard for NER tasks.

We use the spaCy tokenizer (`pt_core_news_sm`) to segment the texts into tokens and align the span annotations with the resulting tokens.

In [2]:
# Mapeamento de labels (mesmo usado no projeto original)
DICT_LABELS = {
    'MULTA_FIXA': 'MULTA',
    'MULTA_PERCENTUAL': 'MULTA',
    'OBRIGACAO_MULTA': 'OBRIGACAO'
}

def translate_label(label):
    """Traduz labels compostos para suas formas simplificadas."""
    return DICT_LABELS.get(label, label)

def load_decicontas(path='dataset/labeled_data/decicontas.json',
                    extra_path='dataset/labeled_data/decicontas_more_ressarcimentos.json'):
    """Carrega o dataset decicontas e opcionalmente dados extras de ressarcimentos."""
    df = pd.read_json(path)
    if os.path.exists(extra_path):
        df_extra = pd.read_json(extra_path)
        df = pd.concat([df[['annotations', 'data']], df_extra[['annotations', 'data']]], ignore_index=True)
    else:
        df = df[['annotations', 'data']].copy()
    return df

def extract_spans(annotations):
    """Extrai spans anotados de uma entrada do Label Studio."""
    spans = []
    for annot in annotations:
        for result in annot.get('result', []):
            val = result.get('value', {})
            if 'start' in val and 'end' in val and 'labels' in val:
                label = translate_label(val['labels'][0])
                spans.append({
                    'start': val['start'],
                    'end': val['end'],
                    'text': val['text'],
                    'label': label
                })
    # Ordenar por posição de início
    spans.sort(key=lambda s: s['start'])
    return spans

In [3]:
def spans_to_bio_tokens(text, spans):
    """
    Converte texto + spans (Label Studio) em listas de tokens e tags BIO.
    Usa tokenização por espaços + pontuação simples para manter alinhamento com offsets.
    """
    import re
    
    # Tokenização baseada em offsets (preserva posições exatas)
    tokens = []
    token_offsets = []
    for match in re.finditer(r'\S+', text):
        tokens.append(match.group())
        token_offsets.append((match.start(), match.end()))
    
    # Atribuir BIO tags
    bio_tags = ['O'] * len(tokens)
    
    for span in spans:
        span_start = span['start']
        span_end = span['end']
        label = span['label']
        first_token = True
        
        for idx, (tok_start, tok_end) in enumerate(token_offsets):
            # Token sobrepõe com o span?
            if tok_start < span_end and tok_end > span_start:
                if first_token:
                    bio_tags[idx] = f'B-{label}'
                    first_token = False
                else:
                    bio_tags[idx] = f'I-{label}'
    
    return tokens, bio_tags

In [4]:
# Carregar dataset
# Ajuste os caminhos conforme necessário
df = load_decicontas(
    path='dataset/labeled_data/decicontas.json',
    extra_path='dataset/labeled_data/decicontas_more_ressarcimentos.json'
)

print(f'Total de registros carregados: {len(df)}')

# Converter para formato BIO
sentencas = []  # Lista de listas de (token, tag)
registros_com_anotacao = 0

for idx, row in df.iterrows():
    text = row['data']['text']
    spans = extract_spans(row['annotations'])
    tokens, bio_tags = spans_to_bio_tokens(text, spans)
    
    if tokens:  # Ignorar textos vazios
        sentencas.append(list(zip(tokens, bio_tags)))
        if any(tag != 'O' for tag in bio_tags):
            registros_com_anotacao += 1

print(f'Total de sentenças/documentos convertidos: {len(sentencas)}')
print(f'Documentos com pelo menos uma entidade anotada: {registros_com_anotacao}')

Total de registros carregados: 1425
Total de sentenças/documentos convertidos: 1425
Documentos com pelo menos uma entidade anotada: 235


In [5]:
# Extrair todos os tokens e labels em listas planas
todos_tokens = []
todos_labels_ner = []
ids_sentenca = []

for i, sentenca in enumerate(sentencas):
    for token_text, ner_tag in sentenca:
        todos_tokens.append(token_text)
        todos_labels_ner.append(ner_tag)
        ids_sentenca.append(i)

print(f'Total de tokens: {len(todos_tokens)}')
print(f'Labels únicas: {sorted(set(todos_labels_ner))}')

# Distribuição de labels
contagem = Counter(todos_labels_ner)
print('\nDistribuição de labels:')
for label, count in contagem.most_common():
    print(f'  {label}: {count} ({100*count/len(todos_labels_ner):.2f}%)')

Total de tokens: 168560
Labels únicas: ['B-MULTA', 'B-OBRIGACAO', 'B-RECOMENDACAO', 'B-RESSARCIMENTO', 'I-MULTA', 'I-OBRIGACAO', 'I-RECOMENDACAO', 'I-RESSARCIMENTO', 'O']

Distribuição de labels:
  O: 144111 (85.50%)
  I-MULTA: 11127 (6.60%)
  I-OBRIGACAO: 7849 (4.66%)
  I-RESSARCIMENTO: 2897 (1.72%)
  I-RECOMENDACAO: 2131 (1.26%)
  B-MULTA: 204 (0.12%)
  B-OBRIGACAO: 120 (0.07%)
  B-RESSARCIMENTO: 63 (0.04%)
  B-RECOMENDACAO: 58 (0.03%)


In [6]:
# Exemplo de sentença anotada
for i, sent in enumerate(sentencas):
    if any(tag != 'O' for _, tag in sent):
        print(f'\nExemplo de sentença #{i} (primeiros 30 tokens):')
        for token, tag in sent[:30]:
            marker = '  <--' if tag != 'O' else ''
            print(f'  {token:40s} {tag}{marker}')
        if len(sent) > 30:
            print(f'  ... ({len(sent)} tokens no total)')
        break


Exemplo de sentença #176 (primeiros 30 tokens):
  Vistos,                                  O
  relatados                                O
  e                                        O
  discutidos                               O
  estes                                    O
  autos,acatando                           O
  o                                        O
  entendimento                             O
  do                                       O
  Ministério                               O
  Público                                  O
  Especial,                                O
  com                                      O
  fulcro                                   O
  nos                                      O
  fundamentos                              O
  jurídicos                                O
  dantes                                   O
  explanados,                              O
  ACORDAM                                  O
  os                                       O
  Cons

# 3. Feature Extraction with BERTimbau

We use the `neuralmind/bert-large-portuguese-cased` model (BERTimbau Large) to extract contextual embeddings for each token. These embeddings serve as features for the linear classifier that feeds Cleanlab.

For each original token, we average the embeddings of its BERT subwords.

In [7]:
hf_model_name = 'neuralmind/bert-large-portuguese-cased'

tokenizer_hf = AutoTokenizer.from_pretrained(hf_model_name)
model_hf = AutoModel.from_pretrained(hf_model_name)
model_hf.to(device)
model_hf.eval()

print(f'Modelo carregado: {hf_model_name}')
print(f'Dimensão dos embeddings: {model_hf.config.hidden_size}')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 52588.52it/s]
BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo carregado: neuralmind/bert-large-portuguese-cased
Dimensão dos embeddings: 1024


In [8]:
from tqdm import tqdm

features_tokens = []

for i_sent, dados_sentenca in enumerate(tqdm(sentencas, desc='Extraindo embeddings')):
    textos_sentenca = [tok for tok, _ in dados_sentenca]

    if not textos_sentenca:
        continue

    inputs = tokenizer_hf(
        textos_sentenca,
        is_split_into_words=True,
        return_tensors='pt',
        padding='longest',
        truncation=True,
        max_length=EFFECTIVE_MAX_LENGTH
    ).to(device)

    word_ids = inputs.word_ids()
    with torch.no_grad():
        outputs = model_hf(**inputs)
        last_hidden_state = outputs.last_hidden_state

    token_subword_embeddings = [[] for _ in range(len(textos_sentenca))]
    for subword_idx, original_word_idx in enumerate(word_ids):
        if original_word_idx is not None:
            embedding = last_hidden_state[0, subword_idx, :]
            token_subword_embeddings[original_word_idx].append(embedding)

    for original_token_idx in range(len(textos_sentenca)):
        if token_subword_embeddings[original_token_idx]:
            stacked = torch.stack(token_subword_embeddings[original_token_idx])
            mean_emb = torch.mean(stacked, dim=0)
            features_tokens.append(mean_emb.cpu().numpy())
        else:
            features_tokens.append(np.zeros(model_hf.config.hidden_size))

features_tokens = np.array(features_tokens)
print(f'Formato da matriz de features: {features_tokens.shape}')

Extraindo embeddings: 100%|██████████| 1425/1425 [01:50<00:00, 12.93it/s]


Formato da matriz de features: (168560, 1024)


# 4. Approach 1: Linear Classifier + Cleanlab (baseline)

Following the original LeNER-Br approach, we train an SGDClassifier (logistic regression) with 5-fold stratified cross-validation to obtain out-of-fold probabilities for each token.

In [9]:
# Codificar labels
label_encoder = LabelEncoder()
labels_codificados = label_encoder.fit_transform(todos_labels_ner)
num_classes = len(label_encoder.classes_)

print(f'Número de classes: {num_classes}')
print(f'Classes: {list(label_encoder.classes_)}')

Número de classes: 9
Classes: ['B-MULTA', 'B-OBRIGACAO', 'B-RECOMENDACAO', 'B-RESSARCIMENTO', 'I-MULTA', 'I-OBRIGACAO', 'I-RECOMENDACAO', 'I-RESSARCIMENTO', 'O']


In [10]:
# Validação cruzada para obter probabilidades out-of-fold
skf = StratifiedKFold(n_splits=NUM_FOLDS_CV, shuffle=True, random_state=RANDOM_SEED)
prob_preditas_sgd = np.zeros((len(features_tokens), num_classes))

print(f'Iniciando validação cruzada de {NUM_FOLDS_CV} folds (SGDClassifier)\n')

for fold_idx, (idx_treino, idx_val) in enumerate(skf.split(features_tokens, labels_codificados)):
    print(f'  Fold {fold_idx + 1}/{NUM_FOLDS_CV}...')
    X_treino = features_tokens[idx_treino]
    X_val = features_tokens[idx_val]
    y_treino = labels_codificados[idx_treino]
    y_val = labels_codificados[idx_val]

    modelo = SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=0.0001,
        max_iter=1000,
        tol=1e-3,
        random_state=RANDOM_SEED,
        class_weight='balanced',
        learning_rate='optimal',
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.1
    )
    modelo.fit(X_treino, y_treino)

    prob_preditas_sgd[idx_val] = modelo.predict_proba(X_val)
    acc = accuracy_score(y_val, modelo.predict(X_val))
    print(f'    Acurácia: {acc:.4f}')

print(f'\nFormato das probabilidades: {prob_preditas_sgd.shape}')

Iniciando validação cruzada de 5 folds (SGDClassifier)

  Fold 1/5...
    Acurácia: 0.8840
  Fold 2/5...
    Acurácia: 0.8894
  Fold 3/5...
    Acurácia: 0.9154
  Fold 4/5...
    Acurácia: 0.9060
  Fold 5/5...
    Acurácia: 0.9082

Formato das probabilidades: (168560, 9)


# 5. Approach 2: Fine-tuned Transformer for NER + Cleanlab

**Improvement over the original**: instead of using only a linear classifier over fixed embeddings, we fine-tune a full Transformer model for the token classification (NER) task. This yields better-calibrated and more accurate probabilities.

We use `neuralmind/bert-base-portuguese-cased` (BERTimbau Base) for fine-tuning, since it is faster and sufficient for this task. K-fold cross-validation is applied in the same way.

In [11]:
# Preparar dados no formato HuggingFace Datasets
label_list = sorted(set(todos_labels_ner))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

# Criar listas de tokens e tags por sentença
all_tokens_by_sent = []
all_tags_by_sent = []
for sent in sentencas:
    toks = [t for t, _ in sent]
    tags = [label2id[tag] for _, tag in sent]
    all_tokens_by_sent.append(toks)
    all_tags_by_sent.append(tags)

print(f'Total de sentenças: {len(all_tokens_by_sent)}')
print(f'Mapeamento label2id: {label2id}')

Total de sentenças: 1425
Mapeamento label2id: {'B-MULTA': 0, 'B-OBRIGACAO': 1, 'B-RECOMENDACAO': 2, 'B-RESSARCIMENTO': 3, 'I-MULTA': 4, 'I-OBRIGACAO': 5, 'I-RECOMENDACAO': 6, 'I-RESSARCIMENTO': 7, 'O': 8}


In [12]:
ft_model_name = 'neuralmind/bert-base-portuguese-cased'
ft_tokenizer = AutoTokenizer.from_pretrained(ft_model_name)

def tokenize_and_align_labels(tokens_list, tags_list):
    """Tokeniza e alinha labels com subpalavras BERT."""
    tokenized = ft_tokenizer(
        tokens_list,
        is_split_into_words=True,
        truncation=True,
        max_length=EFFECTIVE_MAX_LENGTH,
        padding=False
    )
    
    aligned_labels = []
    word_ids = tokenized.word_ids()
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)  # tokens especiais
        elif word_idx != previous_word_idx:
            aligned_labels.append(tags_list[word_idx])
        else:
            # Subpalavra continuação: propagar I- tag ou -100
            label = tags_list[word_idx]
            label_name = id2label[label]
            if label_name.startswith('B-'):
                aligned_labels.append(label2id['I-' + label_name[2:]])
            else:
                aligned_labels.append(label)
        previous_word_idx = word_idx
    
    tokenized['labels'] = aligned_labels
    return tokenized

In [14]:
# Cross-validation com Transformer fine-tuned
from sklearn.model_selection import KFold

# Mapear cada token plano ao seu índice de sentença para reconstruir probabilidades
token_to_sent_and_pos = []  # (sent_idx, pos_in_sent)
for sent_idx, sent in enumerate(sentencas):
    for pos in range(len(sent)):
        token_to_sent_and_pos.append((sent_idx, pos))

prob_preditas_transformer = np.zeros((len(todos_tokens), num_classes))

kf = KFold(n_splits=NUM_FOLDS_CV, shuffle=True, random_state=RANDOM_SEED)
sent_indices = np.arange(len(sentencas))

print(f'Iniciando validação cruzada de {NUM_FOLDS_CV} folds (Transformer Fine-tuned)\n')

for fold_idx, (train_sent_ids, val_sent_ids) in enumerate(kf.split(sent_indices)):
    print(f'\n=== Fold {fold_idx + 1}/{NUM_FOLDS_CV} ===')
    
    # Preparar dados de treino e validação
    train_data = {'input_ids': [], 'attention_mask': [], 'labels': []}
    val_data = {'input_ids': [], 'attention_mask': [], 'labels': []}
    val_word_ids_list = []
    
    for sid in train_sent_ids:
        enc = tokenize_and_align_labels(all_tokens_by_sent[sid], all_tags_by_sent[sid])
        train_data['input_ids'].append(enc['input_ids'])
        train_data['attention_mask'].append(enc['attention_mask'])
        train_data['labels'].append(enc['labels'])
    
    for sid in val_sent_ids:
        enc = tokenize_and_align_labels(all_tokens_by_sent[sid], all_tags_by_sent[sid])
        val_data['input_ids'].append(enc['input_ids'])
        val_data['attention_mask'].append(enc['attention_mask'])
        val_data['labels'].append(enc['labels'])
        val_word_ids_list.append(
            ft_tokenizer(
                all_tokens_by_sent[sid],
                is_split_into_words=True,
                truncation=True,
                max_length=EFFECTIVE_MAX_LENGTH
            ).word_ids()
        )
    
    train_ds = Dataset.from_dict(train_data)
    val_ds = Dataset.from_dict(val_data)
    
    # Modelo
    model_ft = AutoModelForTokenClassification.from_pretrained(
        ft_model_name,
        num_labels=num_classes,
        id2label=id2label,
        label2id=label2id
    )
    
    data_collator = DataCollatorForTokenClassification(ft_tokenizer, padding=True)
    
    training_args = TrainingArguments(
        output_dir=f'./ner_fold_{fold_idx}',
        num_train_epochs=5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        per_device_eval_batch_size=4,
        learning_rate=3e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=50,
        eval_strategy='epoch',
        save_strategy='no',
        report_to='none',
        seed=RANDOM_SEED,
        fp16=torch.cuda.is_available(),
    )
    
    trainer = Trainer(
        model=model_ft,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
    )
    
    trainer.train()
    
    # Predições no conjunto de validação
    predictions = trainer.predict(val_ds)
    logits = predictions.predictions  # (n_samples, seq_len, num_labels)
    
    # Extrair probabilidades por token original (não subpalavra)
    for local_idx, global_sent_idx in enumerate(val_sent_ids):
        word_ids = val_word_ids_list[local_idx]
        sent_logits = logits[local_idx]  # (seq_len, num_labels)
        sent_probs = torch.softmax(torch.tensor(sent_logits), dim=-1).numpy()
        
        # Agregar por token original (primeiro subword)
        n_orig_tokens = len(all_tokens_by_sent[global_sent_idx])
        token_probs = np.zeros((n_orig_tokens, num_classes))
        token_counts = np.zeros(n_orig_tokens)
        
        prev_word_id = None
        for subw_idx, wid in enumerate(word_ids):
            if wid is not None and subw_idx < len(sent_probs):
                if wid != prev_word_id:  # Primeiro subtoken do word
                    token_probs[wid] = sent_probs[subw_idx]
                    token_counts[wid] = 1
            prev_word_id = wid
        
        # Mapear para índice global (plano)
        # Encontrar offset global do primeiro token desta sentença
        global_offset = sum(len(sentencas[s]) for s in range(global_sent_idx))
        for pos in range(n_orig_tokens):
            flat_idx = global_offset + pos
            if flat_idx < len(prob_preditas_transformer):
                # Reordenar para match com label_encoder (que pode ter ordem diferente)
                for cls_idx, cls_name in enumerate(label_list):
                    target_idx = list(label_encoder.classes_).index(cls_name)
                    prob_preditas_transformer[flat_idx, target_idx] = token_probs[pos, cls_idx]
    
    # Limpar modelo
    del model_ft, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

print(f'\nFormato das probabilidades Transformer: {prob_preditas_transformer.shape}')

Iniciando validação cruzada de 5 folds (Transformer Fine-tuned)


=== Fold 1/5 ===


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 42677.44it/s]
BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:

Epoch,Training Loss,Validation Loss
1,6.560945,0.107745
2,1.278252,0.065951
3,0.560233,0.061928
4,0.393599,0.066634
5,0.235411,0.066750



=== Fold 2/5 ===


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 25252.99it/s]
BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:

...

Epoch,Training Loss,Validation Loss
1,6.295891,0.129695
2,1.167808,0.089910
3,0.425389,0.077225
4,0.406391,0.071681
5,0.223685,0.068821



=== Fold 5/5 ===


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/Users/eduardo/Dev/decicontas.br/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/Users/eduardo/Dev/decicontas.br/.venv/lib/python3.12/site-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '429 Too Many Requests' for url 'https://huggingface.co/api/models/neuralmind/bert-base-portuguese-cased'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/eduardo/.pyenv/versions/3.12.3/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/Users/eduardo/Dev/decicontas.br/.venv/lib/python3.12/site-packages/ipykernel/ipkernel.p

Epoch,Training Loss,Validation Loss
1,6.259297,0.120596
2,1.007486,0.087970
3,0.567126,0.085773
4,0.431262,0.084267
5,0.264756,0.086840



Formato das probabilidades Transformer: (168560, 9)


# 6. Ensemble: Combining the Probabilities

**Further improvement**: we combine the probabilities from the two models (SGDClassifier + fine-tuned Transformer) via weighted average. This reduces false positives and increases the robustness of error detection.

In [15]:
# Peso maior para o Transformer (mais expressivo)
PESO_SGD = 0.3
PESO_TRANSFORMER = 0.7

prob_ensemble = PESO_SGD * prob_preditas_sgd + PESO_TRANSFORMER * prob_preditas_transformer

# Renormalizar para somar 1
row_sums = prob_ensemble.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # evitar divisão por zero
prob_ensemble = prob_ensemble / row_sums

print(f'Probabilidades ensemble: {prob_ensemble.shape}')
print(f'Soma de probabilidades (amostra): {prob_ensemble[:5].sum(axis=1)}')

Probabilidades ensemble: (168560, 9)
Soma de probabilidades (amostra): [1. 1. 1. 1. 1.]


# 7. Error Detection with Cleanlab

We apply Confident Learning in three configurations:
1. **SGD only** (baseline, similar to the original LeNER-Br)
2. **Transformer only**
3. **Ensemble** (most robust)

In [16]:
def detectar_erros(labels, pred_probs, nome):
    """Detecta erros de rotulagem usando Cleanlab."""
    print(f'\n--- {nome} ---')
    indices = find_label_issues(
        labels=labels,
        pred_probs=pred_probs,
        return_indices_ranked_by='self_confidence'
    )
    n = len(indices)
    pct = 100 * n / len(labels)
    print(f'Problemas encontrados: {n} ({pct:.2f}% dos tokens)')
    return indices

erros_sgd = detectar_erros(labels_codificados, prob_preditas_sgd, 'SGDClassifier (baseline)')
erros_transformer = detectar_erros(labels_codificados, prob_preditas_transformer, 'Transformer Fine-tuned')
erros_ensemble = detectar_erros(labels_codificados, prob_ensemble, 'Ensemble (SGD + Transformer)')


--- SGDClassifier (baseline) ---
Problemas encontrados: 8295 (4.92% dos tokens)

--- Transformer Fine-tuned ---
Problemas encontrados: 4583 (2.72% dos tokens)

--- Ensemble (SGD + Transformer) ---
Problemas encontrados: 4583 (2.72% dos tokens)


In [17]:
# Interseção: erros detectados por AMBOS os métodos (alta confiança)
erros_sgd_set = set(erros_sgd)
erros_transformer_set = set(erros_transformer)
erros_consenso = erros_sgd_set & erros_transformer_set

print(f'\nErros detectados pelo SGD: {len(erros_sgd_set)}')
print(f'Erros detectados pelo Transformer: {len(erros_transformer_set)}')
print(f'Erros em CONSENSO (ambos detectaram): {len(erros_consenso)}')
print(f'Erros APENAS no SGD: {len(erros_sgd_set - erros_transformer_set)}')
print(f'Erros APENAS no Transformer: {len(erros_transformer_set - erros_sgd_set)}')


Erros detectados pelo SGD: 8295
Erros detectados pelo Transformer: 4583
Erros em CONSENSO (ambos detectaram): 2304
Erros APENAS no SGD: 5991
Erros APENAS no Transformer: 2279


# 8. Analysis of the Detected Errors

We display the problems found by the ensemble with expanded context, showing the problematic token, the original label, the label suggested by the model, and the neighbouring tokens.

In [18]:
def exibir_problemas(indices_problemas, pred_probs, n_exibir=30, janela_contexto=5):
    """
    Exibe os erros de anotação detectados com contexto.
    Foca em erros que NÃO envolvem a classe 'O' (mais informativos).
    """
    problemas = []
    
    for idx in indices_problemas:
        token = todos_tokens[idx]
        label_original = todos_labels_ner[idx]
        label_codificado = labels_codificados[idx]
        
        # Label sugerido pelo modelo
        probs = pred_probs[idx]
        label_sugerido_cod = np.argmax(probs)
        label_sugerido = label_encoder.inverse_transform([label_sugerido_cod])[0]
        confianca = probs[label_sugerido_cod]
        
        # Ignorar se original e sugerido são iguais
        if label_original == label_sugerido:
            continue
        
        sent_idx = ids_sentenca[idx]
        pos_na_sent = idx - sum(len(sentencas[s]) for s in range(sent_idx))
        
        problemas.append({
            'idx_global': idx,
            'token': token,
            'label_original': label_original,
            'label_sugerido': label_sugerido,
            'confianca': confianca,
            'sent_idx': sent_idx,
            'pos_na_sent': pos_na_sent
        })
    
    # Priorizar erros que NÃO são O->entidade (mais prováveis de serem erros reais)
    erros_entidade = [p for p in problemas if p['label_original'] != 'O']
    erros_o = [p for p in problemas if p['label_original'] == 'O']
    problemas_ordenados = erros_entidade + erros_o
    
    print(f'\nTotal de discrepâncias: {len(problemas_ordenados)}')
    print(f'  - Entidade anotada incorretamente: {len(erros_entidade)}')
    print(f'  - Token O que deveria ser entidade: {len(erros_o)}')
    
    print(f'\n{"="*80}')
    print(f'DETALHES DOS PRIMEIROS {min(n_exibir, len(problemas_ordenados))} PROBLEMAS')
    print(f'{"="*80}\n')
    
    for rank, prob in enumerate(problemas_ordenados[:n_exibir], 1):
        sent = sentencas[prob['sent_idx']]
        pos = prob['pos_na_sent']
        
        # Contexto
        inicio = max(0, pos - janela_contexto)
        fim = min(len(sent), pos + janela_contexto + 1)
        
        ctx_antes = ' '.join(f'{t}({tag})' for t, tag in sent[inicio:pos])
        token_destaque = f'**{prob["token"]}**(Orig:{prob["label_original"]}|Sugerido:{prob["label_sugerido"]})'
        ctx_depois = ' '.join(f'{t}({tag})' for t, tag in sent[pos+1:fim])
        
        print(f'Problema #{rank} | Confiança: {prob["confianca"]:.4f} | Sentença #{prob["sent_idx"]}')
        print(f'  {ctx_antes} {token_destaque} {ctx_depois}')
        print()

print('\n=== ERROS DETECTADOS PELO ENSEMBLE ===')
exibir_problemas(erros_ensemble, prob_ensemble, n_exibir=30)


=== ERROS DETECTADOS PELO ENSEMBLE ===

Total de discrepâncias: 4583
  - Entidade anotada incorretamente: 1725
  - Token O que deveria ser entidade: 2858

DETALHES DOS PRIMEIROS 30 PROBLEMAS

Problema #1 | Confiança: 0.3521 | Sentença #1397
  do(I-MULTA) Fundeb;(I-MULTA) h)o(O) dever(O) de(O) **ressarcimento**(Orig:B-RESSARCIMENTO|Sugerido:B-RECOMENDACAO) da(I-RESSARCIMENTO) quantia(I-RESSARCIMENTO) de(I-RESSARCIMENTO) R$(I-RESSARCIMENTO) 107.745,(I-RESSARCIMENTO)

Problema #2 | Confiança: 0.9925 | Sentença #632
  Sr.(O) Tufenis(B-MULTA) da(I-MULTA) Silva(I-MULTA) Morais,(I-MULTA) **para,**(Orig:I-MULTA|Sugerido:O) no(I-MULTA) mérito,(I-MULTA) DAR-LHE(I-MULTA) PROVIMENTO(I-MULTA) PARCIAL,(I-MULTA)

Problema #3 | Confiança: 0.9969 | Sentença #847
  Sr.(O) Paulo(O) Emídio(O) de(O) Medeiros;(O) **b)**(Orig:B-OBRIGACAO|Sugerido:O) Pela(I-OBRIGACAO) fixação(I-OBRIGACAO) de(I-OBRIGACAO) prazo(I-OBRIGACAO) de(I-OBRIGACAO)

Problema #4 | Confiança: 0.9996 | Sentença #632
  no(I-MULTA) item(I-

# 9. Statistical Summary of the Errors

We analyse the distribution of the error types found.

In [19]:
def resumo_erros(indices_problemas, pred_probs, nome):
    """Gera um resumo da distribuição de erros."""
    transicoes = Counter()
    
    for idx in indices_problemas:
        label_orig = todos_labels_ner[idx]
        label_sug = label_encoder.inverse_transform([np.argmax(pred_probs[idx])])[0]
        if label_orig != label_sug:
            transicoes[(label_orig, label_sug)] += 1
    
    print(f'\n=== Matriz de Confusão de Erros ({nome}) ===')
    print(f'{"Original":>20s} -> {"Sugerido":<20s} | Quantidade')
    print('-' * 60)
    for (orig, sug), count in transicoes.most_common(20):
        print(f'{orig:>20s} -> {sug:<20s} | {count}')

resumo_erros(erros_ensemble, prob_ensemble, 'Ensemble')


=== Matriz de Confusão de Erros (Ensemble) ===
            Original -> Sugerido             | Quantidade
------------------------------------------------------------
                   O -> I-OBRIGACAO          | 1208
                   O -> I-RECOMENDACAO       | 616
                   O -> B-RECOMENDACAO       | 413
                   O -> I-MULTA              | 327
             I-MULTA -> I-OBRIGACAO          | 307
                   O -> I-RESSARCIMENTO      | 293
         I-OBRIGACAO -> B-RECOMENDACAO       | 256
         I-OBRIGACAO -> O                    | 232
             I-MULTA -> B-RECOMENDACAO       | 175
      I-RECOMENDACAO -> I-OBRIGACAO          | 109
     I-RESSARCIMENTO -> O                    | 99
             I-MULTA -> O                    | 95
      I-RECOMENDACAO -> O                    | 79
     I-RESSARCIMENTO -> I-OBRIGACAO          | 56
     I-RESSARCIMENTO -> I-MULTA              | 48
             I-MULTA -> I-RESSARCIMENTO      | 48
      I-RECOMENDACAO -

In [20]:
# Exportar erros para análise manual
erros_para_csv = []
for idx in erros_ensemble:
    label_orig = todos_labels_ner[idx]
    probs = prob_ensemble[idx]
    label_sug = label_encoder.inverse_transform([np.argmax(probs)])[0]
    if label_orig != label_sug:
        sent_idx = ids_sentenca[idx]
        pos = idx - sum(len(sentencas[s]) for s in range(sent_idx))
        sent = sentencas[sent_idx]
        ctx_start = max(0, pos - 3)
        ctx_end = min(len(sent), pos + 4)
        contexto = ' '.join(t for t, _ in sent[ctx_start:ctx_end])
        
        erros_para_csv.append({
            'token': todos_tokens[idx],
            'label_original': label_orig,
            'label_sugerido': label_sug,
            'confianca': float(np.max(probs)),
            'sentenca_idx': sent_idx,
            'contexto': contexto
        })

df_erros = pd.DataFrame(erros_para_csv)
df_erros.to_csv('erros_anotacao_decicontas.csv', index=False)
print(f'Exportados {len(df_erros)} erros para erros_anotacao_decicontas.csv')
df_erros.head(10)

Exportados 4583 erros para erros_anotacao_decicontas.csv


,token,label_original,label_sugerido,confianca,sentenca_idx,contexto
0,ressarcimento,B-RESSARCIMENTO,B-RECOMENDACAO,0.352070,1397,h)o dever de ressarcimento da quantia de
1,"para,",I-MULTA,O,0.992528,632,"da Silva Morais, para, no mérito, DAR-LHE"
2,b),B-OBRIGACAO,O,0.996890,847,Emídio de Medeiros; b) Pela fixação de
3,os,I-MULTA,O,0.999632,632,a e mantendo os demais dispositivos do
4,prazo,B-OBRIGACAO,O,0.995486,991,com assinatura de prazo de 90 (noventa)
5,a,I-MULTA,O,0.999222,632,imposta no item a e mantendo os
6,excluindo,I-MULTA,O,0.999557,632,"DAR-LHE PROVIMENTO PARCIAL, excluindo a multa ..."
7,Como,B-RECOMENDACAO,O,0.999082,876,de reexame interposto. Como também para que
8,"mérito,",I-MULTA,O,0.999619,632,"Morais, para, no mérito, DAR-LHE PROVIMENTO PA..."
9,Tufenis,B-MULTA,O,0.990530,632,"interposto pelo Sr. Tufenis da Silva Morais,"


# 10. Conclusion

In this notebook we applied **Confident Learning** techniques (via Cleanlab) to detect annotation errors in the TCE/RN `decicontas.br` dataset.

### Improvements over the original approach (LeNER-Br):

1. **Format adaptation**: automatic conversion of Label Studio annotations (spans with offsets) to token-level BIO format.
2. **Fine-tuned Transformer**: besides the linear classifier over BERT embeddings (baseline), we fine-tuned a full BERTimbau model for token classification, producing more accurate probabilities.
3. **Model ensemble**: weighted combination of the probabilities from two independent models for greater robustness.
4. **Multi-method analysis**: comparison of errors detected by each method and identification of consensus cases.
5. **Structured export**: the errors are exported to CSV to simplify manual review.

### Next steps:
- Manual review of the highest-confidence detected errors.
- Semi-automatic retagging using the ensemble's suggestions as a starting point.
- Use of LLMs (GPT-4, etc.) as additional verifiers for ambiguous cases, leveraging the infrastructure already in place in the project.